# Proton paper: additional parameter-identifiability diagnostics

**New revision analysis for R2.5.** This notebook reproduces the original calibration fit, exposes the full local parameter-correlation matrix, preserves the original 31-point spectrum-slope profile, and profiles the other five parameters while re-optimizing all nuisance coefficients.

Only the original 261 normalized observations at 133, 267, and 400 kilovolts per centimetre are embedded. No observations at 533 kilovolts per centimetre enter this analysis. The fitted model, bounds and reference optimum are unchanged. The parameter-correlation matrix is obtained from the local inverse-information approximation, not correlations of Jacobian columns.

The added intervals use the original fixed residual-variance rule with a nominal one-parameter cutoff of 3.841459. They describe the connected profile branch containing the reference fit. Temporal residual correlation, model discrepancy and unmeasured structure remain outside this working likelihood, so bounded intervals do not establish independent microscopic identification. The original spectrum-slope interval is an accepted-grid summary rather than a root-refined interval. This supplement does not modify earlier blue manuscript revisions or the original 80-check audit.

The code is standalone and writes machine-readable profile ordinates, nuisance optima, numerical diagnostics, correlation coefficients and interval summaries to `proton_R2_5_outputs`.

In [1]:
"""New calibration-only revision analysis for Reviewer 2, comment 5.
All inputs are embedded; no external data files or internet access are needed.
The original fit and its positive spectrum-slope convention are retained.
Generated files are written to proton_R2_5_outputs in the working directory.
"""
from pathlib import Path
import json, time, numpy as np, pandas as pd
from scipy.optimize import least_squares, brentq
from scipy.stats import chi2
import scipy
BASE=Path.cwd()/'proton_R2_5_outputs'
BASE.mkdir(parents=True,exist_ok=True)
PROVENANCE={'source_notebook': 'Hx_NdNiO3_single_cell_evidence_calibrated_PoC_v3_1(7).ipynb', 'source_sha256': '3929eb8def57f4f883706baf2a099b7ad13c8fd6d6bf6cd793ff83afa03640ad', 'embedded_matrix_sha256': '859f76869eb087bc05e0aba24f3543bddb90b03842d10bb49d0dc66d1814354a', 'calibration_fields': [133, 267, 400], 'excluded_field': 533, 'theta_original': [-7.262406628943365, 0.9698094585796684, -0.3065490977982868, -0.12385171386161944, 2.6814065218852408, 1.3412868590757983], 'original_jacobian_condition': 199.80998671880135, 'sse': 0.754192612428207, 'sigma2_fixed': 0.002957618087953753, 'bounds': [[-13.815510557964274, 0.05, -20.0, -20.0, -20.0, -5.0], [0.0, 3.0, 20.0, 20.0, 20.0, 5.0]], 'constants': {'E_A_eV': 0.17403869463781, 'kBT_eV': 0.025851999786435535}, 'original_covariance': [[0.050810551812813916, -0.012368280046818836, -0.007617591115323593, 0.0013208028014118109, 0.2935194967550225, -0.001291785606552064], [-0.012368280046818817, 0.004103884149060378, -0.0027287075775778015, -0.004339524478449726, -0.09512580191096041, 0.003709067431390888], [-0.007617591115323663, -0.0027287075775777837, 0.022252746984101374, 0.018213687513515583, 0.05846974176355433, -0.01407499906506079], [0.0013208028014122014, -0.004339524478449824, 0.01821368751351552, 0.06231255167604741, 0.08749133336182326, -0.05011001953392762], [0.29351949675502176, -0.09512580191096034, 0.05846974176355471, 0.08749133336182102, 2.3844331980742974, -0.07255690918330437], [-0.0012917856065524245, 0.0037090674313909723, -0.014074999065060725, -0.05011001953392759, -0.07255690918330639, 0.04356417030398218]]}
v={
'time': np.array([0.00030539, 0.00033516000000000004, 0.00036784000000000003, 0.0004037, 0.00044306, 0.00048626, 0.00053367, 0.0005857, 0.00064281, 0.00070548, 0.0007742599999999999, 0.00084975, 0.0009326, 0.0010235300000000001, 0.0011233200000000001, 0.00123285, 0.0013530500000000002, 0.00148497, 0.0016297500000000001, 0.0017886500000000001, 0.00196304, 0.00215443, 0.00236449, 0.00259502, 0.00284804, 0.00312572, 0.00343047, 0.00376494, 0.00413201, 0.00497702, 0.00546228, 0.00599484, 0.0065793299999999996, 0.00722081, 0.00792483, 0.00869749, 0.00954548, 0.01047616, 0.01149757, 0.01261857, 0.013848860000000001, 0.01519911, 0.01668101, 0.018307379999999998, 0.020092330000000002, 0.02205131, 0.024201280000000002, 0.026560880000000002, 0.02915053, 0.03199267, 0.03511192, 0.03853529000000001, 0.042292430000000006, 0.04641589, 0.05094138, 0.055908099999999995, 0.06135907, 0.06734151000000001, 0.07390722, 0.08111308, 0.08902151000000001, 0.097701, 0.10722672, 0.1176812, 0.12915497, 0.14174742, 0.15556761, 0.17073526000000003, 0.18738174000000002, 0.20565123, 0.22570197, 0.24770764, 0.27185882, 0.29836471999999997, 0.32745492000000004, 0.35938137000000003, 0.39442061, 0.43287613, 0.47508102, 0.52140083, 0.57223677, 0.62802914, 0.6892612100000001, 0.7564633300000001, 0.8302175700000001, 0.9111627600000001, 1.0],dtype=float),
'y': np.array([[1.0, 1.0, 1.0], [0.894565323087193, 0.93765, 0.96633], [0.948909464101277, 0.88732, 0.96834], [0.8418900344545743, 0.83157, 0.93942], [0.7096591073348034, 0.84634, 0.92828], [0.7160033744983588, 0.77552, 0.89425], [0.6461297079376928, 0.78636, 0.90255], [0.661406934541565, 0.73194, 0.88504], [0.6358616665922034, 0.75305, 0.86113], [0.5687448916034386, 0.74687, 0.85082], [0.6104004982904119, 0.71695, 0.85602], [0.5533861934659831, 0.67133, 0.8208], [0.5723348953090793, 0.69101, 0.81387], [0.6020536082690979, 0.69228, 0.80657], [0.498122212556603, 0.66267, 0.80192], [0.5607317720584177, 0.65516, 0.79982], [0.5959590119342657, 0.62324, 0.77381], [0.5247528915824138, 0.65793, 0.77765], [0.4430264309423629, 0.62053, 0.7521], [0.5490445491601862, 0.61914, 0.73531], [0.5386083085195571, 0.63803, 0.71715], [0.5505478303604985, 0.59148, 0.71122], [0.5071366435305033, 0.59693, 0.72482], [0.4450290538001215, 0.60246, 0.7063], [0.43718150543625384, 0.5947, 0.69197], [0.35453783301489356, 0.57573, 0.67646], [0.4188162448981737, 0.50435, 0.65283], [0.43659806413123753, 0.50099, 0.66998], [0.3178887834723168, 0.48544, 0.61442], [0.5108107468837139, 0.44112, 0.58869], [0.5144848502369245, 0.55736, 0.59425], [0.268719398897769, 0.47917, 0.61706], [0.5227476406106686, 0.51871, 0.5979], [0.51156238748387, 0.52498, 0.5771], [0.32506879088359825, 0.42526, 0.59443], [0.4933626981269432, 0.49004, 0.54544], [0.2302359257925735, 0.43985, 0.51049], [0.1782281781125411, 0.34876, 0.50173], [0.3759911274871814, 0.46189, 0.5125], [0.3122961569089175, 0.36276, 0.50575], [0.31646960191957446, 0.411, 0.46332], [0.20627803869109412, 0.38264, 0.47582], [0.21763149111303257, 0.3491, 0.46022], [0.2920938446942321, 0.34072, 0.45146], [0.2334921590219211, 0.30747, 0.41971], [0.3686455488947467, 0.33149, 0.42929], [0.38684261013768695, 0.27241, 0.43914], [0.05551365173810319, 0.23953, 0.37892], [0.17522424382590324, 0.23716, 0.36286], [0.14942930504781854, 0.26775, 0.36423], [0.19943180175610575, 0.24732, 0.35547], [0.2927613856468183, 0.19733, 0.33266], [0.023957887101479367, 0.28229, 0.31816], [0.01594476755645846, 0.21266, 0.32838], [0.09875663927295855, 0.20399, 0.3094], [0.10134270343573343, 0.19775, 0.31743], [0.0, 0.21881, 0.27053], [0.11728747099219189, 0.17636, 0.27628], [0.15618881422125042, 0.2277, 0.24845], [0.12805748210911405, 0.16408, 0.2604], [0.020701653872131746, 0.12877, 0.27737], [0.0, 0.14153, 0.23859], [0.0, 0.11959, 0.20894], [0.031805435465347004, 0.11372, 0.23431], [0.0, 0.161, 0.20566], [0.06736907393147457, 0.1166, 0.21223], [0.0, 0.08685, 0.2104], [0.004257544658226917, 0.07813, 0.21332], [0.0, 0.06103, 0.1698], [0.0, 0.03139, 0.20246], [0.1412479862076578, 0.09349, 0.16715], [0.0, 0.06156, 0.18257], [0.0, 0.07717, 0.16624], [0.0, 0.07398, 0.15328], [0.10451746513149769, 0.02803, 0.16533], [0.0, 0.0, 0.13832], [0.013608374222406775, 0.012, 0.12518], [0.027463791159550173, 0.04911, 0.12737], [0.0, 0.0, 0.11004], [0.1398288046549155, 0.04223, 0.1021], [0.0, 0.04008, 0.08978], [0.0, 0.0, 0.09945], [0.0, 0.00934, 0.09589], [0.16545554413786037, 0.0, 0.0688], [0.04641512111663307, 0.0, 0.10228], [0.012438863498387652, 0.01959, 0.10046], [0.0015873808478821345, 0.01835, 0.07637]],dtype=float),
'fields': np.array([-1.0, -0.33, 0.335],dtype=float),
'theta': np.array([-7.262406628943365, 0.9698094585796684, -0.3065490977982868, -0.12385171386161944, 2.6814065218852408, 1.3412868590757983],dtype=float),
'jac': np.array([[0.1420376876088004, 0.16172564029693604, 0.08137454092502594, -0.08137453347444534, -0.011562056461048074, -0.14203768205427447], [0.14435623345180412, 0.17153282463550568, 0.08546570688486099, -0.0854656845331192, -0.01188515034937175, -0.14435623347523294], [0.14605827673349872, 0.18153230845928192, 0.0895107164978981, -0.08951070159673691, -0.012156923327493772, -0.14605827427664086], [0.1471349689742038, 0.1916775107383728, 0.09347779303789139, -0.09347778558731079, -0.012370323285501209, -0.14713496677124022], [0.1475985975062245, 0.20192743837833405, 0.09733860194683075, -0.09733858704566956, -0.012519148367867796, -0.14759859262246755], [0.14748136361422262, 0.21223827451467514, 0.10106506943702698, -0.10106504708528519, -0.012598069202326789, -0.1474813641171922], [0.14683499268915526, 0.2225676327943802, 0.10463143140077591, -0.10463140904903412, -0.012602928989654493, -0.14683499644609432], [0.14572796221759704, 0.23287785053253174, 0.10801556706428528, -0.10801555961370468, -0.012530935227698626, -0.14572795807752228], [0.1442410277852316, 0.24313823133707047, 0.11119972914457321, -0.11119971424341202, -0.012380759741306157, -0.14424102691726237], [0.14246285680096507, 0.2533203959465027, 0.11416898667812347, -0.11416898667812347, -0.012152635933642058, -0.14246285202310882], [0.14048302152693018, 0.26340747624635696, 0.11691399663686752, -0.11691397428512573, -0.011848214298515734, -0.14048302121410447], [0.13838688183597633, 0.2733898460865021, 0.11942918598651886, -0.11942917108535767, -0.011470481840716945, -0.1383868839966726], [0.1362508342192756, 0.2832646891474724, 0.12171243876218796, -0.12171244621276855, -0.011023584245032955, -0.13625082998437202], [0.1341383835742999, 0.29303737729787827, 0.12376496940851212, -0.12376496940851212, -0.010512564686992519, -0.1341383838732949], [0.13209875617357145, 0.3027185946702957, 0.12559016048908234, -0.12559015303850174, -0.009943105422218934, -0.1320987611939919], [0.1301662502631974, 0.312326118350029, 0.12719348818063736, -0.12719348073005676, -0.009321136002546006, -0.13016624617383968], [0.12836285110921258, 0.32187799364328384, 0.12858089804649353, -0.12858088314533234, -0.00865291663429124, -0.12836284693157887], [0.12669957083693753, 0.33139659464359283, 0.12975917011499405, -0.12975915521383286, -0.007944465784779853, -0.12669956752657974], [0.12517998371920774, 0.34090376645326614, 0.13073494285345078, -0.13073494285345078, -0.0072017157844547555, -0.1251799852502277], [0.12380257858958045, 0.35042132437229156, 0.13151467591524124, -0.13151466101408005, -0.006430273866493368, -0.12380257808724385], [0.12256307227093496, 0.3599689304828644, 0.13210429251194, -0.1321042850613594, -0.0056355166395172475, -0.12256306884140653], [0.12145568683427886, 0.3695647493004799, 0.132509283721447, -0.132509283721447, -0.004822492836273359, -0.12145568052041635], [0.12047403393376056, 0.3792252279818058, 0.13273466005921364, -0.13273465633392334, -0.003995923313634069, -0.12047403343614269], [0.11961175138425104, 0.3889629878103733, 0.1327848769724369, -0.1327848695218563, -0.0031603747156663193, -0.11961175067785505], [0.11886235645611545, 0.3987899422645569, 0.13266396895051003, -0.13266395777463913, -0.002320024681162107, -0.11886235812581794], [0.11821960699577526, 0.40871354937553406, 0.1323755495250225, -0.1323755383491516, -0.0014789938874267415, -0.11821960662970747], [0.11767723161119725, 0.41873960569500923, 0.1319228671491146, -0.131922859698534, -0.0006411154257229827, -0.11767723037479894], [0.11722898917471353, 0.42887112125754356, 0.13130880519747734, -0.13130879774689674, 0.0001899498864510274, -0.11722898854296888], [0.11686863855865622, 0.4391072392463684, 0.130536001175642, -0.13053599372506142, 0.001010627368479456, -0.11686863753869377], [0.11638601426434428, 0.45987679809331894, 0.1285232938826084, -0.12852328643202782, 0.002607469014646193, -0.11638601427048191], [0.11625054788760271, 0.47039251402020454, 0.12728748843073845, -0.12728748098015785, 0.0033771979200932704, -0.11625054380106119], [0.1161764514881922, 0.48097700253129005, 0.1259012408554554, -0.1259012334048748, 0.004123672645925965, -0.11617645109742135], [0.11615643648363284, 0.4916117452085018, 0.12436626851558685, -0.12436625733971596, 0.004843957591622996, -0.11615643715198412], [0.11618283470449033, 0.5022732689976692, 0.12268427386879921, -0.12268427014350891, 0.0055352019042408385, -0.11618283634035226], [0.11624753427495793, 0.5129331424832344, 0.12085699662566185, -0.12085698544979095, 0.006194631142586404, -0.11624753309930515], [0.11634191651934932, 0.5235578268766403, 0.11888619139790535, -0.11888618394732475, 0.006819584788439982, -0.11634191471099646], [0.11645679543336786, 0.5341080911457539, 0.11677373945713043, -0.11677373200654984, 0.007407499574900593, -0.11645679631315017], [0.11658236125901894, 0.5445388667285442, 0.11452164128422737, -0.11452162638306618, 0.007955923379431533, -0.11658235979625747], [0.1167081143133705, 0.554798286408186, 0.11213218420743942, -0.11213217303156853, 0.008462502720119384, -0.11670811491997471], [0.11682281574484009, 0.5648280717432499, 0.10960786044597626, -0.10960786044597626, 0.008925028602724283, -0.11682281321371893], [0.11691443726357119, 0.5745625793933868, 0.10695156455039978, -0.10695155709981918, 0.009341419849025274, -0.1169144368670673], [0.11697011702727358, 0.5839288979768753, 0.10416651517152786, -0.10416651144623756, 0.009709760608043252, -0.11697011818515297], [0.11697612219548334, 0.5928462147712708, 0.10125645995140076, -0.10125645622611046, 0.010028307302563735, -0.11697612292426418], [0.11691783354090242, 0.6012257412075996, 0.09822569787502289, -0.09822569414973259, 0.010295510858009707, -0.1169178308500432], [0.11677973339494817, 0.6089708358049393, 0.09507906064391136, -0.09507905133068562, 0.010510050740403187, -0.1167797329600863], [0.11654541975402519, 0.6159766484051943, 0.09182212129235268, -0.09182211942970753, 0.010670849544063034, -0.11654541898564302], [0.11619764577708692, 0.6221304852515459, 0.08846120908856392, -0.08846120163798332, 0.010777107029566501, -0.1161976468266198], [0.11571837569776802, 0.6273122280836105, 0.08500341884791851, -0.08500341512262821, 0.010828332077753975, -0.11571837310298154], [0.11508889946990374, 0.631394749507308, 0.08145680464804173, -0.08145679906010628, 0.01082436769721092, -0.11508889896825918], [0.1142899569027231, 0.6342449709773064, 0.07783031836152077, -0.07783031649887562, 0.010765429924795388, -0.11428995482025404], [0.11330193232740192, 0.6357249598950148, 0.07413393445312977, -0.07413393259048462, 0.010652137001033632, -0.1133019308252499], [0.112105098250851, 0.6356935705989599, 0.07037869095802307, -0.07037868909537792, 0.010485537850863447, -0.11210509772132426], [0.11067991056151662, 0.6340084597468376, 0.06657669879496098, -0.06657669693231583, 0.010267148205552567, -0.11067990955496418], [0.1090073755489289, 0.6305285785347223, 0.06274116225540638, -0.06274116039276123, 0.009998968663657848, -0.10900737446398538], [0.10706949437950726, 0.6251172330230474, 0.058886393904685974, -0.058886390179395676, 0.009683511782464072, -0.10706949489545516], [0.10484977348712747, 0.6176456520333886, 0.055027762427926064, -0.05502776149660349, 0.009323815623703328, -0.10484977309251851], [0.10233381806245441, 0.6079971622675657, 0.05118165258318186, -0.05118165165185928, 0.00892344826879482, -0.10233381795632965], [0.09950999350885789, 0.596071919426322, 0.047365360893309116, -0.04736535996198654, 0.008486508513497144, -0.09950999356898552], [0.09637017320306511, 0.5817922241985798, 0.04359698947519064, -0.043596986681222916, 0.0080176088489275, -0.09637017395201498], [0.09291049959265191, 0.5651081465184689, 0.03989523742347956, -0.03989523556083441, 0.007521846980818361, -0.09291050060226812], [0.08913222108268623, 0.5460035894066095, 0.036279214546084404, -0.036279212683439255, 0.007004758940488784, -0.08913222196110958], [0.08504252071508601, 0.5245024776086211, 0.03276817314326763, -0.03276817221194506, 0.006472261081379315, -0.08504252177401168], [0.08065529816904078, 0.5006746873259544, 0.029381190426647663, -0.02938118949532509, 0.005930563594843537, -0.08065529953999477], [0.07599185923711312, 0.47464146465063095, 0.026136803440749645, -0.026136802043765783, 0.005386072564177062, -0.07599186069556298], [0.07108150918633259, 0.44658014830201864, 0.023052630946040154, -0.023052629083395004, 0.004845272047393557, -0.0710815113255101], [0.06596185828722241, 0.41672710375860333, 0.020144918467849493, -0.02014491753652692, 0.004314585146769412, -0.06596186020518521], [0.06067889131924851, 0.38537897262722254, 0.01742810197174549, -0.017428101040422916, 0.003800223269299694, -0.0606788930959465], [0.05528662132596248, 0.3528911315370351, 0.014914336148649454, -0.014914335682988167, 0.003308024706875356, -0.05528662313991819], [0.04984633117846607, 0.3196732345968485, 0.012613064143806696, -0.012613063445314765, 0.002843289482704453, -0.04984633332410125], [0.04442527605486907, 0.28618100890889764, 0.010530611500144005, -0.010530610801652074, 0.0024106171500055357, -0.04442527821939366], [0.039094848951681374, 0.25290421210229397, 0.008669856935739517, -0.008669856004416943, 0.0020137566602843496, -0.039094851070082835], [0.0339282016931076, 0.22035067505203187, 0.007030000211670995, -0.007029999862425029, 0.0016554811088451986, -0.03392820373550987], [0.02899739092766353, 0.18902683758642524, 0.005606463877484202, -0.005606463528238237, 0.0013374921766871296, -0.028997392824627835], [0.02437012998273023, 0.15941532549913973, 0.004390937741845846, -0.004390937625430524, 0.0010603714614287989, -0.024370131786914707], [0.020106397300000044, 0.13195122592151165, 0.0033716075122356415, -0.003371608443558216, 0.0008235814758711198, -0.020106398838860097], [0.01625510575230995, 0.10699848813237622, 0.0025335546815767884, -0.0025335546233691275, 0.0006255229351022055, -0.01625510733855831], [0.01285116318510367, 0.08482874825131148, 0.0018593254499137402, -0.0018593253334984183, 0.00046365043780932104, -0.012851164562352782], [0.009913273184514593, 0.06560498499311507, 0.0013296520337462425, -0.0013296518009155989, 0.0003346401379721686, -0.00991327434735346], [0.0074427727866798925, 0.04937217099359259, 0.0009242723026545718, -0.0009242722735507414, 0.00023460167350395768, -0.007442773773046825], [0.005423764617842318, 0.03605675511062145, 0.0006227996200323105, -0.0006227996200323105, 0.00015931572104911878, -0.0054237653264855535], [0.003824646883849634, 0.025475825670582708, 0.0004055656827404164, -0.00040556566091254354, 0.00010448330017304177, -0.003824647493875263], [0.0026009885710170312, 0.01735566687420942, 0.0002543751397752203, -0.0002543751252233051, 6.595255290904508e-05, -0.0026009890496715184], [0.0016994700081508988, 0.01135793344292324, 0.00015309656009776518, -0.00015309654918382876, 3.9920494115500534e-05, -0.001699470352834672], [0.0010624518309284414, 0.0071104541420936584, 8.80509614944458e-05, -8.80509614944458e-05, 2.3074959326997757e-05, -0.001062452764061402], [0.0006325863402331025, 0.004238695837557316, 4.81712631881237e-05, -4.81712631881237e-05, 1.2679314343303579e-05, -0.0006325865587586719], [0.00035687778014102005, 0.002393764676526189, 2.494105137884617e-05, -2.494093496352434e-05, 6.589428080888157e-06, -0.000356877865395124], [0.00018968833538304304, 0.001273447007406503, 1.2152624549344182e-05, -1.2152609997428954e-05, 3.2208421123951163e-06, -0.00018968839055946916], [0.10071748295028769, 0.08412948250770569, 0.04504618048667908, -0.01486525684595108, -0.007512011399232746, -0.033236769796088944], [0.10634873338809508, 0.09078335016965866, 0.048371002078056335, -0.015962444245815277, -0.008000071310224116, -0.0350950837939957], [0.11193303917874947, 0.09782594442367554, 0.05184774845838547, -0.017109766602516174, -0.008497395103977387, -0.03693789989909928], [0.11740418291692098, 0.10525359958410263, 0.055466167628765106, -0.01830386370420456, -0.008999803752396009, -0.038743382191468036], [0.12269435806162465, 0.11306318640708923, 0.05921553075313568, -0.01954112946987152, -0.009502754229590446, -0.040489144825941834], [0.12773011864082592, 0.12124468386173248, 0.06308093667030334, -0.020816728472709656, -0.010000847698066287, -0.042150941099264094], [0.13243598122386466, 0.12978246808052063, 0.06704417616128922, -0.022124603390693665, -0.01048799900387975, -0.04370387439654455], [0.13673775502754715, 0.1386563628911972, 0.0710843950510025, -0.023457862436771393, -0.01095755337822017, -0.04512345915811425], [0.1405656493996186, 0.1478424370288849, 0.07517853379249573, -0.024808920919895172, -0.01140243370369266, -0.046386670736779934], [0.14385542181282276, 0.15730765461921692, 0.07929924875497818, -0.026168763637542725, -0.011814976576380802, -0.04747228979544192], [0.14655512220176192, 0.1670176163315773, 0.08341873437166214, -0.027528196573257446, -0.012187513035149972, -0.048363190884812696], [0.14862664824349933, 0.17693306505680084, 0.08750735968351364, -0.028877444565296173, -0.01251233243972894, -0.0490467979370405], [0.15004906417863353, 0.18701036274433136, 0.09153430908918381, -0.0302063450217247, -0.012781954774402337, -0.049516189670966625], [0.15082147028349155, 0.19720452278852463, 0.09546925872564316, -0.03150486946105957, -0.012989450188057932, -0.049771088346576696], [0.15096406981724386, 0.2074689418077469, 0.09928248077630997, -0.03276324272155762, -0.013128669618742487, -0.0498181430598153], [0.1505181310632091, 0.21776032447814941, 0.1029471606016159, -0.03397258371114731, -0.013194564333709042, -0.04967098529058888], [0.1495443407422004, 0.2280348688364029, 0.1064380556344986, -0.03512457758188248, -0.013183330417090456, -0.04934963453913494], [0.14811930463117012, 0.23825585842132568, 0.10973449796438217, -0.03621239215135574, -0.013092647730282592, -0.0488793762563639], [0.14633092173191944, 0.24839157611131668, 0.11281952261924744, -0.03723044693470001, -0.012921743819953776, -0.048289212054578756], [0.14427246784416317, 0.2584182247519493, 0.11568070948123932, -0.03817465156316757, -0.01267138558221747, -0.047609921082174635], [0.14203672222684338, 0.2683192566037178, 0.11830981820821762, -0.03904224932193756, -0.012343854255150315, -0.046872121398338316], [0.1397099619169642, 0.27808698266744614, 0.12070280313491821, -0.03983192890882492, -0.011942737023109126, -0.04610428690934545], [0.1373672086853763, 0.2877223938703537, 0.12285932153463364, -0.040543586015701294, -0.011472676941911276, -0.04533118646967967], [0.13506967144143284, 0.2972323074936867, 0.12478170543909073, -0.04117797315120697, -0.010939195107851879, -0.04457300066878685], [0.1328627805881119, 0.306631438434124, 0.1264747753739357, -0.041736677289009094, -0.010348284980774192, -0.04384472746744423], [0.13077766533340682, 0.3159372806549072, 0.12794450670480728, -0.04222169518470764, -0.00970628456805071, -0.04315663213658383], [0.12883249352968387, 0.3251712918281555, 0.129197858273983, -0.04263530671596527, -0.00901952909844482, -0.042514724970117115], [0.12703552375797814, 0.3343563452363014, 0.13024205714464188, -0.0429798886179924, -0.008294231541919213, -0.041921722265384875], [0.12538819024143086, 0.34351468086242676, 0.131084106862545, -0.04325776547193527, -0.00753641036579911, -0.04137810729001214], [0.12252777119713067, 0.3618375137448311, 0.13218820467591286, -0.04362211748957634, -0.00594555384364831, -0.040434166190092574], [0.12130279587746408, 0.3710399866104126, 0.13246218115091324, -0.04371253401041031, -0.00512293689246337, -0.04002992115394266], [0.12020565877562156, 0.3802911005914211, 0.13255784660577774, -0.043744102120399475, -0.004288651172334638, -0.03966786760337931], [0.11922945910336309, 0.38960424438118935, 0.13247988745570183, -0.0437183752655983, -0.003447146625746572, -0.039345722515484204], [0.11836744944609645, 0.39898981526494026, 0.13223251327872276, -0.0436367392539978, -0.0026026522957596433, -0.03906125841534191], [0.11761300549851196, 0.4084555059671402, 0.13181952387094498, -0.04350045695900917, -0.0017591832725331056, -0.038812292266439286], [0.11695957733382473, 0.4180063232779503, 0.13124434277415276, -0.04331064224243164, -0.0009205434719340974, -0.03859666047406421], [0.11640065965236483, 0.4276443086564541, 0.13051006197929382, -0.04306832328438759, -9.03645360651403e-05, -0.038412221662705706], [0.11592971894191839, 0.43736863136291504, 0.12961944937705994, -0.04277442395687103, 0.0007279025025631836, -0.038256806682851356], [0.11554017295951391, 0.4471748284995556, 0.12857507541775703, -0.04242978245019913, 0.0015309052520635813, -0.03812825749458932], [0.11522528916626414, 0.45705535635352135, 0.12737925723195076, -0.042035166174173355, 0.0023154622050088254, -0.038024346623004156], [0.11497816113142027, 0.46699878945946693, 0.1260341815650463, -0.04159129038453102, 0.0030784904945949775, -0.03794279104517756], [0.1147916326149811, 0.4769899509847164, 0.12454186007380486, -0.04109882190823555, 0.0038170378486461166, -0.0378812383031876], [0.11465824883693532, 0.48700931668281555, 0.12290427461266518, -0.04055841639637947, 0.004528242299782292, -0.037837222065706465], [0.11457019030602218, 0.4970327988266945, 0.12112335488200188, -0.039970722049474716, 0.005209344689160501, -0.0378081593506003], [0.11451921793350975, 0.507031686604023, 0.11920103803277016, -0.039336349815130234, 0.005857698391606565, -0.03779134219272875], [0.11449660788786616, 0.5169719271361828, 0.11713932082056999, -0.03865598514676094, 0.006470756811874148, -0.037783882095942134], [0.11449310081218066, 0.5268139243125916, 0.114940345287323, -0.037930313497781754, 0.007046067827426545, -0.037782726697482255], [0.1144988341140582, 0.5365123525261879, 0.11260635778307915, -0.037160102277994156, 0.0075813210247914955, -0.037784615329580135], [0.11450329631259384, 0.5460154488682747, 0.11013990640640259, -0.03634617477655411, 0.008074293516683597, -0.03778608735165642], [0.11449527266510627, 0.5552649796009064, 0.10754377767443657, -0.03548944741487503, 0.00852291801792743, -0.03778344326671939], [0.1144627948975142, 0.5641957819461823, 0.1048210933804512, -0.03459096699953079, 0.008925271731021124, -0.03777272250216379], [0.11439310991406028, 0.5727354474365711, 0.10197542235255241, -0.033651892095804214, 0.009279586071268234, -0.037749725628971983], [0.11427264850703527, 0.5808041673153639, 0.09901078790426254, -0.03267356753349304, 0.009584281399391572, -0.037709972700711795], [0.11408700817277387, 0.588314563035965, 0.09593175537884235, -0.03165748156607151, 0.009837980219926466, -0.03764871297443702], [0.11382095362461017, 0.5951716061681509, 0.09274349920451641, -0.030605359002947807, 0.010039532883355, -0.037560913801086844], [0.11345843961939042, 0.6012726947665215, 0.08945188298821449, -0.02951912395656109, 0.010188037731022035, -0.037441284233387065], [0.11298265276333352, 0.6065079160034657, 0.08606351725757122, -0.02840096317231655, 0.010282877217053595, -0.03728427641455017], [0.11237608306930154, 0.6107604615390301, 0.08258581534028053, -0.027253322303295135, 0.010323740137229724, -0.0370841064087764], [0.11162064193652871, 0.6139072831720114, 0.0790270958095789, -0.026078946888446808, 0.010310654972293765, -0.03683481252665679], [0.11069779885322062, 0.615820087492466, 0.07539659179747105, -0.024880878627300262, 0.010244016979391157, -0.03653027476697419], [0.10958878350095962, 0.6163666006177664, 0.07170451804995537, -0.023662494495511055, 0.010124624313987808, -0.03616429952740759], [0.10827483864168932, 0.6154122389853001, 0.06796212680637836, -0.022427504882216454, 0.009953699564090692, -0.035730697371736245], [0.10673752506966652, 0.6128222141414881, 0.06418171897530556, -0.02117997221648693, 0.009732920314948006, -0.03522338162754471], [0.10495909170874165, 0.6084640715271235, 0.06037663482129574, -0.01992429420351982, 0.009464439988617483, -0.03463649892402963], [0.10292293086022652, 0.6022108402103186, 0.05656127445399761, -0.018665222451090813, 0.009150910767491513, -0.03396456667330688], [0.10061408295095248, 0.5939445784315467, 0.05275103356689215, -0.01740784291177988, 0.008795487414884673, -0.0332026480466206], [0.0980198315614693, 0.5835606064647436, 0.0489622438326478, -0.01615754421800375, 0.008401836652839455, -0.03234654430930319], [0.09513035385342108, 0.5709721418097615, 0.045212067663669586, -0.014919985085725784, 0.007974122574428463, -0.03139301701278683], [0.09193944460601398, 0.5561155304312706, 0.041518363170325756, -0.013701061718165874, 0.0075169962239702535, -0.030340017130115318], [0.08844527372282018, 0.5389557769522071, 0.0378994969651103, -0.012506835162639618, 0.007035553917892745, -0.02918694057675736], [0.08465117015578208, 0.51949233841151, 0.03437411691993475, -0.011343460530042648, 0.0065352924396613644, -0.027934886508819905], [0.08056639544187061, 0.4977649273350835, 0.030960879288613796, -0.010217091999948025, 0.0060220402692036535, -0.026586910760811332], [0.0762068769727493, 0.47385910246521235, 0.027678134385496378, -0.009133785497397184, 0.005501874745625283, -0.02514826990967906], [0.07159581847575279, 0.447911127936095, 0.024543552193790674, -0.00809937296435237, 0.004981016306400916, -0.023626620900444904], [0.06676417456169915, 0.42011192068457603, 0.02157373446971178, -0.007119333371520042, 0.004465711264802526, -0.022032177969313217], [0.06175087815646558, 0.39070936385542154, 0.01878378400579095, -0.0061986492946743965, 0.003962092532117915, -0.02037779075804993], [0.05660275490156531, 0.360008514020592, 0.016186856664717197, -0.005341663490980864, 0.003476033567009543, -0.018678909943655772], [0.05137406886807129, 0.3283692265395075, 0.013793727150186896, -0.004551930585876107, 0.0030129892133116316, -0.016953443507908818], [0.046125606418372204, 0.29620058252476156, 0.011612370843067765, -0.0038320827297866344, 0.002577839142773305, -0.015221450876173647], [0.04092326545073817, 0.26395181589759886, 0.009647600585594773, -0.0031837085261940956, 0.0021747363340307384, -0.013504678570814766], [0.03583611242065761, 0.23209944274276495, 0.007900777505710721, -0.0026072568725794554, 0.0018069697051194416, -0.011825917937413333], [0.03093394477704355, 0.20113079412840307, 0.006369633018039167, -0.0021019791020080447, 0.0014768546623249157, -0.01020820264971306], [0.026284410653858843, 0.1715242835925892, 0.005048219813033938, -0.0016659126849845052, 0.0011856567752637503, -0.008673856234958364], [0.021949849514034316, 0.14372747321613133, 0.003927026002202183, -0.001295918715186417, 0.0009335609733967295, -0.007243451069602903], [0.017984032606796378, 0.1181341785704717, 0.002993252535816282, -0.0009877734119072556, 0.0007196916692369728, -0.005934731355610016], [0.014429086870649446, 0.09506254340521991, 0.0022312672808766365, -0.0007363181794062257, 0.0005421876505485147, -0.0047615992826064305], [0.011312905518770617, 0.07473621156532317, 0.0016232169000431895, -0.0005356615874916315, 0.00039833128566976923, -0.003733259328256668], [0.054236400722848466, 0.03896038979291916, 0.02138577401638031, 0.007164232432842255, -0.0038576347170002742, 0.018169190706814674], [0.05851506588829318, 0.04243723303079605, 0.02324359118938446, 0.007786594331264496, -0.00417924204945184, 0.019602551373098305], [0.06303268514517847, 0.046193189918994904, 0.025240808725357056, 0.00845567137002945, -0.0045222641222891205, 0.021115956711530232], [0.06778176729366361, 0.050242915749549866, 0.027382753789424896, 0.009173229336738586, -0.004886923224240657, 0.022706879287979476], [0.0727535479787446, 0.05460350215435028, 0.029675588011741638, 0.009941324591636658, -0.0052734360868167745, 0.02437243616109663], [0.07793285192953077, 0.05929025262594223, 0.03212409466505051, 0.010761573910713196, -0.005681658222343923, 0.026107511359816555], [0.08329852076400143, 0.06431735306978226, 0.0347319170832634, 0.011635199189186096, -0.006111075588137353, 0.027905000294571058], [0.08882360996385799, 0.0696982592344284, 0.037501685321331024, 0.012563064694404602, -0.0065608212581557595, 0.029755909743694072], [0.09447535501947117, 0.075445756316185, 0.0404350608587265, 0.01354575902223587, -0.007029669865783066, 0.031649235647815306], [0.10021140531174696, 0.08156826347112656, 0.04353071004152298, 0.014582790434360504, -0.007515670827426334, 0.03357081882100097], [0.10598419346574503, 0.08807379007339478, 0.04678642004728317, 0.015673451125621796, -0.00801650400448143, 0.03550470032202399], [0.11173834082254974, 0.09496699273586273, 0.05019746720790863, 0.016816161572933197, -0.008529168499113062, 0.03743234378232373], [0.11741083389590856, 0.1022481843829155, 0.053756169974803925, 0.018008321523666382, -0.009049954868163722, 0.03933262429640539], [0.12293293564384287, 0.1099141463637352, 0.05745229125022888, 0.019246526062488556, -0.009574467352026534, 0.04118252832667629], [0.1282304173244635, 0.11795622855424881, 0.0612722784280777, 0.020526215434074402, -0.010097543294981155, 0.042957192587047915], [0.13322711532832446, 0.1263626590371132, 0.06520026922225952, 0.021842092275619507, -0.01061344486922215, 0.04463108721539717], [0.13784442950809675, 0.13511361926794052, 0.06921611726284027, 0.02318739891052246, -0.011115586771166525, 0.04617788245835952], [0.1420071462476311, 0.14418593794107437, 0.07329757511615753, 0.024554692208766937, -0.011596911333683135, 0.047572392851429744], [0.14564497117252523, 0.15355009585618973, 0.07741930335760117, 0.02593546360731125, -0.01204986351861073, 0.04879107159618753], [0.14869687593370695, 0.16317182779312134, 0.08155375719070435, 0.027320511639118195, -0.01246657986217721, 0.04981344925357205], [0.15111434726658204, 0.17301137000322342, 0.08567120879888535, 0.028699859976768494, -0.012839044077109604, 0.0506233058067427], [0.15286536136903758, 0.18302540481090546, 0.08974099159240723, 0.030063234269618988, -0.013159389921026705, 0.051209898271942736], [0.15393724003679654, 0.19316831976175308, 0.09373222291469574, 0.031400300562381744, -0.013420148492649574, 0.05156897722695193], [0.15433849525083373, 0.20339148491621017, 0.09761407226324081, 0.03270071744918823, -0.01361447052053007, 0.051703392284317966], [0.15409988996902693, 0.21364832669496536, 0.10135789215564728, 0.03395489603281021, -0.013736504253889384, 0.05162346426297571], [0.15327349326241363, 0.22389224916696548, 0.10493678599596024, 0.0351538211107254, -0.013781570514991747, 0.051346629681028814], [0.15193020787043388, 0.2340814471244812, 0.10832756012678146, 0.03628972917795181, -0.013746418331182264, 0.05089662420010737], [0.15015554960436164, 0.24417965859174728, 0.11151088774204254, 0.03735615313053131, -0.01362934163646939, 0.05030211058969682], [0.14804439468458439, 0.2541566863656044, 0.11447160691022873, 0.038347989320755005, -0.013430268187016352, 0.0495948734170444], [0.1432000557028467, 0.2736702784895897, 0.11968784779310226, 0.040095433592796326, -0.012793616596823362, 0.04797201629374015], [0.14064672814716878, 0.2831893861293793, 0.12193508446216583, 0.0408482626080513, -0.012363257282542745, 0.04711665481661121], [0.13810678564823173, 0.29255205392837524, 0.12394218891859055, 0.041520632803440094, -0.011864883174547123, 0.04626577606331459], [0.1356368492427938, 0.3017694354057312, 0.12571322172880173, 0.04211392253637314, -0.01130444049632494, 0.045438349676833384], [0.13327769682886492, 0.31085794419050217, 0.12725397944450378, 0.04263008385896683, -0.010688333941867252, 0.0446480238014749], [0.13105547989578226, 0.3198379650712013, 0.12857145816087723, 0.0430714413523674, -0.010023121028665987, 0.043903586131294574], [0.12898422469446635, 0.32873231172561646, 0.12967322021722794, 0.04344053566455841, -0.009315314816467368, 0.04320971380813478], [0.1270689417727154, 0.33756455034017563, 0.13056692481040955, 0.0437399223446846, -0.00857123941959835, 0.042568095491283035], [0.1253085539243074, 0.3463579788804054, 0.1312599554657936, 0.043972089886665344, -0.007796907748165943, 0.041978364563920345], [0.12369846660822806, 0.3551342189311981, 0.13175925239920616, 0.044139351695775986, -0.006998031233189068, 0.04143898790116726], [0.12223220017257459, 0.36391326040029526, 0.13207123056054115, 0.04424386844038963, -0.0061799448041526825, 0.040947788021310924], [0.12090257991095342, 0.3727125972509384, 0.13220176473259926, 0.04428758844733238, -0.005347665239799023, 0.040502365250626976], [0.11970225414737733, 0.38154739886522293, 0.13215616717934608, 0.04427231475710869, -0.004505887000213942, 0.04010025603518775], [0.11862398303012406, 0.3904300890862942, 0.13193925097584724, 0.04419964924454689, -0.0036590308524862766, 0.03973903292726744], [0.11766069290500276, 0.39937036857008934, 0.13155538588762283, 0.04407105594873428, -0.0028112744354080246, 0.03941632680454325], [0.11680543527892645, 0.40837543457746506, 0.13100850954651833, 0.04388785362243652, -0.001966538366428809, 0.03912981853789502], [0.1160513878458227, 0.4174494259059429, 0.13030216842889786, 0.04365123063325882, -0.0011285612641017608, 0.03887721399480384], [0.11539177703037676, 0.4265935644507408, 0.12943962961435318, 0.04336228221654892, -0.0003009080839108385, 0.038656241261951975], [0.11481982156294375, 0.4358060993254185, 0.12842381745576859, 0.04302198067307472, 0.0005130521106020242, 0.03846463953562225], [0.11432869041721039, 0.4450818598270416, 0.12725745141506195, 0.04263124614953995, 0.0013100669574773777, 0.0383001091284955], [0.11391143356122366, 0.4544123262166977, 0.12594302371144295, 0.04219091683626175, 0.002087038307561197, 0.03816032535665123], [0.11356092809707971, 0.4637852907180786, 0.1244828850030899, 0.041701771318912506, 0.0028409930491092596, 0.03804291076556624], [0.11326982645243408, 0.473184559494257, 0.1228792667388916, 0.04116455838084221, 0.0035690678254370616, 0.03794538791431214], [0.11303049328699445, 0.48258979618549347, 0.12113432958722115, 0.04058000445365906, 0.004268514592138028, 0.037865212704357075], [0.11283494547674541, 0.49197619780898094, 0.11925020441412926, 0.03994882106781006, 0.004936700617083513, 0.03779970494456215], [0.11267479927954806, 0.5013141296803951, 0.11722905561327934, 0.03927173838019371, 0.005571093198072717, 0.03774605668338184], [0.11254120980640943, 0.5105688683688641, 0.11507311835885048, 0.03854949772357941, 0.006169269387964553, 0.03770130443486256], [0.112424816648216, 0.51970025151968, 0.11278476193547249, 0.03778289631009102, 0.006728928498418632, 0.03766231251424181], [0.11231568539882462, 0.528662346303463, 0.11036651954054832, 0.036972783505916595, 0.007247878206849732, 0.03762575359614716], [0.11220325995021543, 0.5374030843377113, 0.10782117024064064, 0.03612009435892105, 0.007724063731823409, 0.03758808927279521], [0.1120763158135556, 0.5458640195429325, 0.10515180230140686, 0.03522585332393646, 0.008155565054446902, 0.03754556449919137], [0.11192291446617374, 0.5539800003170967, 0.10236182436347008, 0.034291211515665054, 0.008540621925851082, 0.03749417704172823], [0.11173037719083741, 0.561678908765316, 0.09945511072874069, 0.03331746160984039, 0.008877637340451848, 0.03742967331208533], [0.1114852581455975, 0.5688815321773291, 0.09643598645925522, 0.03230605833232403, 0.009165207405998015, 0.03734756086552987], [0.11117333641298043, 0.5755014345049858, 0.09330933541059494, 0.03125862590968609, 0.009402136625921402, 0.037243068128614554], [0.11077962754148342, 0.5814449451863766, 0.09008067660033703, 0.030177026987075806, 0.009587458738905139, 0.037111173561879145], [0.11028840637208721, 0.5866113528609276, 0.08675617910921574, 0.02906332165002823, 0.009720471451497494, 0.036946615380750954], [0.10968326731050906, 0.5908931717276573, 0.0833427906036377, 0.027919836342334747, 0.009800750331157413, 0.036743894332944976], [0.10894720434823628, 0.5941766891628504, 0.07984825223684311, 0.02674916572868824, 0.009828186317477453, 0.036497313970766], [0.1080627339153311, 0.5963427033275366, 0.07628119178116322, 0.025554200634360313, 0.009803006561752092, 0.03620101536861038], [0.10701205338363196, 0.5972675941884518, 0.07265113480389118, 0.0243381317704916, 0.009725809854243836, 0.035849036837069165], [0.1057772511219667, 0.5968246981501579, 0.06896858662366867, 0.023104477673768997, 0.009597591631665176, 0.03543537919183144], [0.10434056682098962, 0.5948860831558704, 0.06524504721164703, 0.021857092157006264, 0.009419766900703725, 0.03495409046438875], [0.10268471362913731, 0.5913247410207987, 0.06149302050471306, 0.02060016244649887, 0.009194198718765558, 0.03439938116414105], [0.10079325979140445, 0.5860172361135483, 0.057726021856069565, 0.0193382166326046, 0.008923214865629853, 0.033765743706602916], [0.09865108184539009, 0.5788468793034554, 0.05395854730159044, 0.018076112493872643, 0.008609623125798983, 0.03304811350261187], [0.0962448869380764, 0.5697074076160789, 0.05020603258162737, 0.016819020733237267, 0.008256717540368995, 0.032242038379737194], [0.09356379682548763, 0.5585071500390768, 0.0464847581461072, 0.015572394244372845, 0.007868276323072793, 0.031343872305939914], [0.09059999996717086, 0.5451737251132727, 0.0428117411211133, 0.014341933652758598, 0.007448550745843704, 0.030351001053334558], [0.08734945183847914, 0.5296591119840741, 0.03920458443462849, 0.013133535161614418, 0.007002237005398271, 0.029262067861950816], [0.08381261102148983, 0.5119450688362122, 0.03568126820027828, 0.011953224427998066, 0.006534442879926976, 0.028077225488796594], [0.07999518157962587, 0.4920486584305763, 0.03225992154330015, 0.010807073675096035, 0.006050626599693888, 0.026798387562571076], [0.07590883722236241, 0.4700277056545019, 0.02895853342488408, 0.009701108559966087, 0.005556530855070383, 0.025429461885276776], [0.07157186801367646, 0.44598584482446313, 0.025794618763029575, 0.008641197346150875, 0.0050580900604561985, 0.023976577484681332], [0.06700971929575499, 0.42007683543488383, 0.022784863132983446, 0.007632929366081953, 0.004561326677427146, 0.022448257160301438], [0.062255337100299554, 0.3925076490268111, 0.01994470926001668, 0.006681477651000023, 0.004072228087619059, 0.020855539341834154], [0.057349266379474594, 0.36353986896574497, 0.01728794351220131, 0.00579146109521389, 0.003596610788207712, 0.019212005168808326], [0.05233942601086981, 0.3334888811223209, 0.014826264698058367, 0.0049667987041175365, 0.00313997364661536, 0.017533708708405947]],dtype=float),
'corr': np.array([[1.0000000000000002, -0.856514674870871, -0.2265420816681992, 0.023473251812818514, 0.8432714917620242, -0.027456726766900502], [-0.8565146748708696, 1.0000000000000002, -0.28554044746941537, -0.27136687185990177, -0.9616304456435752, 0.2773974466377155], [-0.2265420816682013, -0.2855404474694135, 1.0000000000000002, 0.48912324016630765, 0.2538322391399791, -0.45205558180284183], [0.023473251812825453, -0.2713668718599079, 0.48912324016630593, 1.0, 0.22697848210374744, -0.9617718919728346], [0.8432714917620222, -0.9616304456435746, 0.25383223913998076, 0.22697848210374164, 1.0, -0.22512383099639355], [-0.027456726766908166, 0.27739744663772176, -0.4520555818028398, -0.961771891972834, -0.2251238309963998, 1.0]],dtype=float),
'se': np.array([0.22541196022574736, 0.06406156530292073, 0.1491735465292066, 0.24962482183478352, 1.5441610013448395, 0.20872031598285343],dtype=float),
'lo': np.array([-13.815510557964274, 0.05, -20.0, -20.0, -20.0, -5.0],dtype=float),
'hi': np.array([0.0, 3.0, 20.0, 20.0, 20.0, 5.0],dtype=float),
'EA': np.array(0.17403869463781,dtype=float),
'kBT': np.array(0.025851999786435535,dtype=float)
}
t=v['time'];y=v['y'];fields=v['fields'];theta=v['theta'];lo=v['lo'];hi=v['hi'];se=v['se']
z=np.linspace(0,1,21);z2=(z-.5)**2;scale=float(v['EA']/v['kBT'])
def predjac(th):
 lt=th[0]+th[1]*scale*z[None,:]+th[5]*fields[:,None]
 logits=(th[2]+th[3]*fields[:,None])*z[None,:]+th[4]*z2
 w=np.exp(logits-logits.max(axis=1,keepdims=True));w/=w.sum(axis=1,keepdims=True)
 u=t[None,:,None]*np.exp(-lt)[:,None,:];g=np.exp(-u);wg=w[:,None,:]*g;p=wg.sum(axis=2)
 ez=(w*z).sum(axis=1);ez2=(w*z2).sum(axis=1)
 dl=(wg*u).sum(axis=2);dc=(wg*u*z).sum(axis=2)*scale
 db=(wg*(z[None,None,:]-ez[:,None,None])).sum(axis=2)
 db2=(wg*(z2[None,None,:]-ez2[:,None,None])).sum(axis=2)
 jac=np.stack([dl,dc,db,fields[:,None]*db,db2,fields[:,None]*dl],axis=2).reshape(-1,6)
 return p.reshape(-1),jac
def residual(th):return predjac(th)[0]-y.T.reshape(-1)
sse0=float(residual(theta)@residual(theta));sigma2=sse0/(y.size-6);cutoff=float(chi2.ppf(.95,1))
assert np.max(np.abs(predjac(theta)[1]-v['jac']))<1e-6
cov=np.asarray(PROVENANCE['original_covariance'])

def run_profile(j):
 started=time.time();free=np.delete(np.arange(6),j);cache={}
 def profile(a):
  key=float(a)
  if key in cache:return cache[key]
  base=theta.copy();base[j]=a
  starts=[theta[free],np.clip(theta+cov[:,j]/cov[j,j]*(a-theta[j]),lo+1e-9,hi-1e-9)[free]]
  if cache:starts.append(min(cache.items(),key=lambda kv:abs(kv[0]-a))[1]['theta'][free])
  fits=[]
  def unpack(x):q=base.copy();q[free]=x;return q
  for start in starts:
   r=least_squares(lambda x:residual(unpack(x)),np.clip(start,lo[free]+1e-9,hi[free]-1e-9),jac=lambda x:predjac(unpack(x))[1][:,free],bounds=(lo[free],hi[free]),xtol=1e-10,ftol=1e-10,gtol=1e-10,max_nfev=4000)
   fits.append((float(r.fun@r.fun),unpack(r.x),r))
  s,q,r=min(fits,key=lambda a:a[0]);entry={'value':float(a),'sse':s,'theta':q,'stat':float((s-sse0)/sigma2),'success':bool(r.success),'optimality':float(r.optimality),'nfev':int(r.nfev),'active_nuisance':int(np.sum(np.abs(r.active_mask)>0)),'start_sse_spread':max(f[0] for f in fits)-s}
  cache[key]=entry;return entry
 limL=max(lo[j],theta[j]-5*se[j]);limR=min(hi[j],theta[j]+5*se[j])
 grid=np.unique(np.r_[np.linspace(limL,theta[j],31),np.linspace(theta[j],limR,31)])
 for a in sorted(grid,key=lambda a:abs(a-theta[j])):profile(a)
 endpoints=[]
 for side in [-1,1]:
  seq=sorted([a for a in cache if side*(a-theta[j])>=0],reverse=side<0);prev=theta[j];found=None
  for a in seq:
   if profile(a)['stat']>=cutoff:found=(min(prev,a),max(prev,a));break
   prev=a
  if found is None:
   bound=lo[j] if side<0 else hi[j]
   for a in np.linspace(prev,bound,31)[1:]:
    if profile(a)['stat']>=cutoff:found=(min(prev,a),max(prev,a));break
    prev=a
  endpoints.append(float(prev) if found is None else float(brentq(lambda a:profile(a)['stat']-cutoff,*found,xtol=2e-7)))
 name=['log_tau0','chi_cond','b0','b1','b2','cF'][j]
 summary={'parameter':name,'estimate':float(theta[j]),'lower':endpoints[0],'upper':endpoints[1],'local_se':float(se[j]),'lower_at_bound':bool(abs(endpoints[0]-lo[j])<1e-6),'upper_at_bound':bool(abs(endpoints[1]-hi[j])<1e-6),'cutoff':cutoff,'sse0':sse0,'sigma2':sigma2}
 rows=[]
 for a,e in sorted(cache.items()):
  d={k:v for k,v in e.items() if k!='theta'};d['parameter']=name;d.update({f'fit_{k}':float(v) for k,v in enumerate(e['theta'])});rows.append(d)
 pd.DataFrame(rows).to_csv(BASE/f'profile_{name}.csv',index=False)
 (BASE/f'interval_{name}.json').write_text(json.dumps(summary,indent=2))
 print(summary,'seconds',time.time()-started,'points',len(cache),flush=True)
 return summary

# Verify the reference fit using the original prediction formula and optimizer.
def original_prediction(th, k):
    log_tau=th[0]+th[1]*scale*z+th[5]*fields[k]
    tau=np.exp(np.clip(log_tau,-50.0,50.0))
    logits=(th[2]+th[3]*fields[k])*z+th[4]*z2
    w=np.exp(logits-logits.max());w/=w.sum()
    return np.exp(-t[:,None]/tau[None,:])@w

def original_residual(th):
    return np.concatenate([original_prediction(th,k)-y[:,k] for k in range(3)])
reference_fit=least_squares(original_residual,
    x0=np.array([np.log(7e-4),1.,0.,0.,0.,1.]),bounds=(lo,hi),
    max_nfev=20000,xtol=1e-11,ftol=1e-11,gtol=1e-11)
assert np.max(np.abs(reference_fit.x-theta))<1e-5
assert abs(float(reference_fit.fun@reference_fit.fun)-sse0)<1e-10
assert np.max(np.abs(original_residual(theta)-residual(theta)))<1e-12

# Correlation comes from the inverse information matrix, not from correlations
# between Jacobian columns. Positive rescaling chi_cond -> alpha preserves it.
names=['log_tau0','alpha_A','b0','b1','b2','cF']
correlation=cov/np.sqrt(np.outer(np.diag(cov),np.diag(cov)))
pd.DataFrame(correlation,index=names,columns=names).to_csv(BASE/'parameter_correlation.csv')
print('Parameter correlation (order: log_tau0, alpha_A, b0, b1, b2, cF)')
print(pd.DataFrame(correlation,index=names,columns=names).round(6))
print('Original-coordinate Jacobian condition:',np.linalg.cond(v['jac']))
print('Alpha-coordinate Jacobian condition:',np.linalg.cond(v['jac']@np.diag([1,1/scale,1,1,1,1])))

# Preserve the original 31-point alpha/chi profile and its accepted-grid summary.
# It is a fixed-residual-scale diagnostic, not a concentrated unknown-variance
# likelihood ratio. We do not silently substitute refined alpha endpoints.
old_rows=[]
free=np.array([0,2,3,4,5])
for a in np.linspace(max(.05,theta[1]-.5),min(3.,theta[1]+.5),31):
    def unpack(x):
        q=theta.copy();q[1]=a;q[free]=x;return q
    fit=least_squares(lambda x:original_residual(unpack(x)),theta[free],
          bounds=(lo[free],hi[free]),max_nfev=3000)
    old_rows.append({'chi_cond':float(a),'alpha_A':float(a*scale),
       'sse':float(fit.fun@fit.fun),'success':bool(fit.success)})
old=pd.DataFrame(old_rows)
old['stat']=(old.sse-old.sse.min())/sigma2
old.to_csv(BASE/'original_alpha_profile_31_points.csv',index=False)
inside=old[old.stat<=cutoff]
print('Retained original accepted-grid alpha interval:',inside.alpha_A.min(),inside.alpha_A.max())

# Newly executed profiles for the other five parameters.
intervals=[run_profile(j) for j in [0,2,3,4,5]]
pd.DataFrame(intervals).to_csv(BASE/'additional_profile_intervals.csv',index=False)
print(pd.DataFrame(intervals)[['parameter','estimate','lower','upper']].to_string(index=False))

checks={
 'reference_fit_reproduced':True,
 'calibration_observations':int(y.size),
 'calibration_fields_kV_per_cm':[133,267,400],
 'holdout_observations_used':False,
 'residual_variance_fixed_at_reference_fit':float(sigma2),
 'nominal_one_parameter_cutoff':float(cutoff),
 'intervals_are_connected_component_near_reference':True,
 'all_additional_endpoints_inside_original_bounds':all(not r['lower_at_bound'] and not r['upper_at_bound'] for r in intervals),
 'source_notebook_sha256':PROVENANCE['source_sha256'],
 'numpy_version':np.__version__, 'scipy_version':scipy.__version__,
 'interpretation':'Working Gaussian fixed-scale profile support; not global uniqueness or independent physical identification.'
}
(BASE/'revision_checks.json').write_text(json.dumps(checks,indent=2))
(BASE/'provenance.json').write_text(json.dumps(PROVENANCE,indent=2))
print('All new profile endpoints crossed the nominal threshold before parameter bounds.')


Parameter correlation (order: log_tau0, alpha_A, b0, b1, b2, cF)
          log_tau0   alpha_A        b0        b1        b2        cF
log_tau0  1.000000 -0.856515 -0.226542  0.023473  0.843271 -0.027457
alpha_A  -0.856515  1.000000 -0.285540 -0.271367 -0.961630  0.277397
b0       -0.226542 -0.285540  1.000000  0.489123  0.253832 -0.452056
b1        0.023473 -0.271367  0.489123  1.000000  0.226978 -0.961772
b2        0.843271 -0.961630  0.253832  0.226978  1.000000 -0.225124
cF       -0.027457  0.277397 -0.452056 -0.961772 -0.225124  1.000000
Original-coordinate Jacobian condition: 199.80998671880135
Alpha-coordinate Jacobian condition: 76.61971765570888


Retained original accepted-grid alpha interval: 5.855659291559125 7.426486658273314


{'parameter': 'log_tau0', 'estimate': -7.262406628943365, 'lower': -7.809293678400705, 'upper': -6.878065869211527, 'local_se': 0.22541196022574736, 'lower_at_bound': False, 'upper_at_bound': False, 'cutoff': 3.841458820694124, 'sse0': 0.7541926124282072, 'sigma2': 0.002957618087953754} seconds 2.470531940460205 points 68


{'parameter': 'b0', 'estimate': -0.3065490977982868, 'lower': -0.6345952895190254, 'upper': -0.042341204460107924, 'local_se': 0.1491735465292066, 'lower_at_bound': False, 'upper_at_bound': False, 'cutoff': 3.841458820694124, 'sse0': 0.7541926124282072, 'sigma2': 0.002957618087953754} seconds 3.019679307937622 points 68


{'parameter': 'b1', 'estimate': -0.12385171386161944, 'lower': -0.842524813421617, 'upper': 0.4508389008287892, 'local_se': 0.24962482183478352, 'lower_at_bound': False, 'upper_at_bound': False, 'cutoff': 3.841458820694124, 'sse0': 0.7541926124282072, 'sigma2': 0.002957618087953754} seconds 1.9911243915557861 points 67


{'parameter': 'b2', 'estimate': 2.6814065218852408, 'lower': -0.6350929348348473, 'upper': 5.784621911200614, 'local_se': 1.5441610013448395, 'lower_at_bound': False, 'upper_at_bound': False, 'cutoff': 3.841458820694124, 'sse0': 0.7541926124282072, 'sigma2': 0.002957618087953754} seconds 2.2224152088165283 points 68


{'parameter': 'cF', 'estimate': 1.3412868590757983, 'lower': 0.8848582004044124, 'upper': 1.9164391976893667, 'local_se': 0.20872031598285343, 'lower_at_bound': False, 'upper_at_bound': False, 'cutoff': 3.841458820694124, 'sse0': 0.7541926124282072, 'sigma2': 0.002957618087953754} seconds 2.056774139404297 points 67


parameter  estimate     lower     upper
 log_tau0 -7.262407 -7.809294 -6.878066
       b0 -0.306549 -0.634595 -0.042341
       b1 -0.123852 -0.842525  0.450839
       b2  2.681407 -0.635093  5.784622
       cF  1.341287  0.884858  1.916439
All new profile endpoints crossed the nominal threshold before parameter bounds.
